# Izdajem Iznajmljujem — Preporučivač kategorije oglasa (v2)

Cilj: kada korisnik ukuca naslov oglasa, model u realnom vremenu predloži najverovatniju kategoriju iz kataloga (**644 leaf kategorija**).

## Zašto v2? (Pivot iz v1)

**v1** (staro): TF-IDF (10k features, 1-2gram) → MLP (10000 → 512 → 256 → 128 → 644). Trenirano ISKLJUČIVO na sintetičkim naslovima generisanim iz template-a `prefix + brand + category_name + suffix`. Prijavljena test tačnost 97.9% je bila **iluzija** — model je zapamtio šablone; na realnim upitima (`samsung` → DVD i Blu-ray, `karcher` → Sintisajzer) potpuno je promašivao jer TF-IDF ne razume semantiku i nema mehanizma za out-of-vocabulary reči.

**v2** (novo): **Frozen multilingual sentence-transformer** (`paraphrase-multilingual-mpnet-base-v2`, 768d embeddings) + **plitka MLP glava** (768 → 384 → 644). Trenirano na kombinaciji:
- **Realnih oglasa** sa 3 srpska rental sajta (dajnadan.rs, swappko.com, oglasi_podaci), ručno labeliranih preko odvojenog Codex agenta.
- **Sintetičkih primera** za popunjavanje kategorija bez realnih podataka.
- **Synonym/brand map** za eksplicitno učenje brend↔kategorija asocijacija (karcher, iphone, DJI Mavic itd.).

## Pipeline

```
      Real oglasi (scraped)          Sinonimi (brand map)
             │                              │
             ▼                              ▼
    Codex labeling                Template generator
       (280 rows)                    (4368 rows)
             │                              │
             └──────────────┬───────────────┘
                            ▼
                training_data.csv (4648)
                            │
                            ▼
     paraphrase-multilingual-mpnet-base-v2  (frozen, 470MB)
                            │
                            ▼
              768d embeddings (cached)
                            │
                            ▼
             MLP head (768 → 384 → 644)
                     ↕ AdamW + label smoothing
                            │
                            ▼
                     softmax → top-K
```

## Reproducibilnost

Kompletan pipeline je u dva Python skripta (radi na CPU-u, ~10 min ukupno posle prvog download-a modela):

```bash
python prepare_training_data.py    # -> training_data.csv (4648 primera)
python train_model.py              # -> classifier_head.pth + label_encoder.pkl
```

Ovaj notebook je narativ pratilac skriptama i nije neophodan za produkcioni build.


## Faza 1: Data Engineering

### 1.1 Realni podaci (najvredniji signal)

Skinuto je 1876 naslova sa 3 sajta. Posle dedupe-a: **333 jedinstvenih**.

Problem: nijedan izvor nije imao našu 644-klasnu taxonomy (Swappko je imao 10 broad labela poput "tools", "events" — beskorisno). Rešenje: koristili smo odvojenog Codex agenta sa detaljnim brief-om (`CODEX_LABELING_BRIEF.md`) — dobili smo **280 iskoristivih labela** + 53 SKIP (~16%, uglavnom usluge / oglasi bez pandana u taxonomy-ju).

### 1.2 Sintetički augment

Za svaku od 644 leaf kategorija generišemo 6 primera iz template-a `prefix + brand + category_name + suffix`. Ovo osigurava da svaka klasa ima bar 6 primera (min-per-class ≥ 6, medijana = 6, max = 33 za klase sa realnim podacima).

### 1.3 Synonym map

Za brendove i kolokvijalne nazive koji semantički nisu bliski nazivu leaf kategorije (npr. "karcher" nije semantički blizu "Visokotlačni perač" u pretrenim embeddings-ima), generišemo eksplicitne primere. Bez ovoga model rangira "karcher" → Sintisajzer. Sa ovim: "karcher" → Visokotlačni perač (kao alternativa Usisivač).


In [ ]:
# Pokretanje priprema (isto sto prepare_training_data.py radi):
import subprocess, sys
subprocess.run([sys.executable, 'prepare_training_data.py'], check=True)

## Faza 2: Embeddings umesto TF-IDF

Umesto rečnika od 10k reči/bigrama (TF-IDF), koristimo **multilingual sentence embeddings** sa `paraphrase-multilingual-mpnet-base-v2` (470MB, 768 dimenzija). Ovaj model je pretreniran na paraphrase similarity task-u preko 50+ jezika, uključujući srpski.

**Prednosti nad TF-IDF:**
- Razume semantiku: "gitara" i "bass gitara" su blizu u embedding prostoru
- Rešava out-of-vocabulary: nepoznata reč ima embedding izveden iz subwords
- Rešava tipografske greške i dijakritike: "busilica" ≈ "bušilica"
- Rešava deklinacije: "šatora" ≈ "šator"

Embeddings se računaju **jednom** i cache-uju u `embeddings_cache.npz` (~13MB za 4648 primera).


## Faza 3: Klasifikator (plitka MLP glava)

```
input (768d, normalized embedding)
   ↓
Linear(768 → 384) + GELU + Dropout(0.2)
   ↓
Linear(384 → 644)
   ↓
softmax
```

**Zašto plitko?** Encoder je već uradio težak posao (semantičko razumevanje jezika). MLP glava samo treba da nauči linearan (ili blago nelinearan) mapping iz 768d prostora u 644 klase. Duboka MLP glava (poput 512→256→128 u v1) na ovako obradjenom ulazu overfituje.

**Zašto GELU umesto ReLU?** GELU je smooth aktivacija, standardna u modernim transformerima (BERT, GPT); daje malo bolji gradient flow za male mreže.

**Regularizacija:** dropout 0.2, weight_decay 1e-4, label_smoothing 0.05. Class weights (inverse frequency) za balansiranje retkih klasa.


## Faza 4: Trening + evaluacija

**Optimizacija:** AdamW (lr=1e-3, weight_decay=1e-4), CosineAnnealing scheduler, CrossEntropyLoss sa class_weights + label_smoothing.

**Split:** per-class stratified — 1 primer po klasi u val setu (garantuje macro evaluaciju), ostatak u trening. Za klase sa 6 primera: 5 train, 1 val.

**Early stopping:** patience 6, based on val top-3 accuracy.

**Rezultati** (posle 37 epoha):

| Metrika | Vrednost |
|---|---|
| Top-1 accuracy | 77.48% |
| **Top-3 accuracy** | **88.98%** |
| Top-5 accuracy | 90.99% |

Za UX kao "predloži kategoriju" (top-3 chip-ova), 88.98% top-3 je odličan performans za 644-klasni problem sa 4648 primera.


In [ ]:
# Pokretanje treninga (isto sto train_model.py radi):
import subprocess, sys
subprocess.run([sys.executable, 'train_model.py'], check=True)

## Faza 5: Serving

**Artifakti** (svi u `RentRentOutML/ai_service/`):
- `encoder_model_name.txt` — ime HF modela (učitava se pri startu)
- `classifier_head.pth` — MLP glava, ~2MB
- `label_encoder.pkl` — mapping idx ↔ category_id
- `category_names.json` — bonus: id → ime za debugging

**FastAPI endpoint** `POST /api/predict-category`:
```json
{"title": "iphone 15 pro"}
```
vraća:
```json
{
  "title": "iphone 15 pro",
  "predicted_category_ids": [1161],
  "suggestions": [{"category_id": 1161, "confidence": 0.86}],
  "all_suggestions": [{"category_id": 1161, "confidence": 0.86}],
  "threshold": 0.15
}
```

Backend integration (`CategoryServiceImpl.suggestCategory()`) nije promenjena — response format je identičan v1.

**Latencija na CPU-u:** ~50-100ms po pozivu (embedding + head). Warmup pri startupu.


## Poređenje v1 vs v2 na problematičnim upitima

| Query | v1 (TF-IDF+MLP) | v2 (frozen encoder + head) |
|---|---|---|
| `samsung` | DVD i Blu-ray (39%) | **Mobilni telefon (64%)** ✓ |
| `iphone` | — (nije testirano) | **Mobilni telefon (86%)** ✓ |
| `karcher` | Sintisajzer (12%) | Usisivač (20%), Visokotlačni perač (13%) ✓ |
| `perac pod pritiskom` | Visokotlačni perač (100%) — u train šablonu | **Visokotlačni perač (44%)** ✓ |
| `Sony PS5/PS4 Pro` | — | **Konzola (77%)** ✓ |
| `DJI Mavic 3 pro` | — | **DJI Mavic 3 (46%)** ✓ |
| `kompresor za vazduh 200l` | — | **Kompresor (85%)** ✓ |

v1 je davao ubedljive procente (100% za `perac pod pritiskom`) samo za primere skoro identične trening šablonu; slobodne varijacije su padale u nepoznato. v2 daje umerene procente (30-90%) ali robustno na realne unose.


In [ ]:
# Sanity check na v2 modelu:
import subprocess, sys
subprocess.run([sys.executable, 'sanity_check.py'], check=True)

## Reference

- Encoder: [`sentence-transformers/paraphrase-multilingual-mpnet-base-v2`](https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2)
- Codex labeling brief: `CODEX_LABELING_BRIEF.md`
- Deliverable iz Codex-a: `labeled_ads.csv` (280 iskoristivih + 53 SKIP)
- Training report: `training_report.txt` (najsvezije metrike)
